# 基于 mindspore 的口罩佩戴检测



## 1. 实验介绍

### 1.1 实验要求

本实训重点涵盖：
- OpenCV DNN模块的人脸检测技术
- MindSpore框架下的MobileNetV2模型构建
- 动态学习率调整和早停策略
- 边训练边验证的模型优化方法

### 1.2 实验环境

可以使用基于 Python 的 OpenCV 、PIL 库进行图像相关处理，使用 Numpy 库进行相关数值运算，使用 MindSpore 深度学习框架训练模型等。


## 2. OpenCV 人脸检测

数据信息存放在 `/datasets/5f680a696ec9b83bb0037081-momodel/data` 文件夹下。    
该文件夹主要有文件夹 `image`、文件 `train.txt` 、文件夹 `keras_model_data` 和文件夹 `mindspore_model_data`共四部分：
+ **image 文件夹**：图片分成两类，戴口罩的和没有戴口罩的  
+ **train.txt**：  存放的是 image 文件夹下对应图片的标签  （keras 框架专用文件）
+ **keras_model_data** 文件夹：存放 keras 框架相关预训练好的模型 （keras 框架专用文件夹）
+ **mindspore_model_data** 文件夹：存放 mindspore 框架相关预训练好的模型（mindspore 框架专用文件）

opencv 人脸检测模型在数据集 **mindspore_model_data/opencv_dnn** 文件夹中

In [ ]:
import os
# 数据集路径
basic_path = "/home/jovyan/work/datasets/5f680a696ec9b83bb0037081-momodel/data/"
# opencv 人脸检测模型在数据集 mindspore_model_data/opencv_dnn 文件夹中
opencv_dnn_path = basic_path + 'mindspore_model_data/opencv_dnn'
print(opencv_dnn_path)
# 查看文件夹里面文件
os.listdir(opencv_dnn_path)


发现该文件夹下有我们需要的路径，所以
+ **依赖文件夹的路径为 opencv_dnn_path** =           
`/datasets/5f680a696ec9b83bb0037081-momodel/data/mindspore_model_data/opencv_dnn`

+ **deploy.prototxt 文件的路径**：       
`opencv_dnn_path + '/' + 'deploy.prototxt'`
+ **res10_300x300_ssd_iter_140000_fp16.caffemodel 文件的路径**：        
`opencv_dnn_path + '/' + 'res10_300x300_ssd_iter_140000_fp16.caffemodel'`

In [ ]:
import os
import numpy as np
import cv2
import matplotlib.pyplot as plt

class FaceDet():
    def __init__(self):
        self.opencv_dnn_path = '/home/jovyan/work/datasets/5f680a696ec9b83bb0037081-momodel/data/mindspore_model_data/opencv_dnn/'
        self.threshold = 0.15
        self.caffe_model = self.opencv_dnn_path + "deploy.prototxt"
        self.caffe_param = self.opencv_dnn_path + "res10_300x300_ssd_iter_140000_fp16.caffemodel"

    def draw_detections(self, image, detections):
        h, w, c = image.shape
        for i in range(0, detections.shape[2]):
            confidence = detections[0, 0, i, 2]
            if confidence > self.threshold:
                box = detections[0, 0, i, 3:7] * np.array([w, h, w, h])
                (startX, startY, endX, endY) = box.astype("int")
                text = "{:.2f}%".format(confidence * 100)
                y = startY - 10 if startY - 10 > 10 else startY + 10
                cv2.rectangle(image, (startX, startY), (endX, endY),
                              (0, 255, 0), 1)
                cv2.putText(image, text, (startX, y),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 255, 0), 2)
        return image

    def detect(self, image):
        net = cv2.dnn.readNetFromCaffe(self.caffe_model, self.caffe_param)

        # def blobFromImage(image, scalefactor=None, size=None, mean=None, swapRB=None, crop=None, ddepth=None)
        # image：输入图像
        # mean：对每个通道像素值减去对应的均值，这里用(104.0, 177.0, 123.0)，和模型训练时的值一致
        # scalefactor：对像素值的缩放比例
        # size：模型输入图片的尺寸
        # swapRB：OpenCV默认的图片通道顺序是BGR，如果需要交换R和G，则设为True
        # crop: 调整图片大小后，是否裁剪
        blob = cv2.dnn.blobFromImage(image, 1.0, (300, 300), (104.0, 177.0, 123.0), False, False)
        net.setInput(blob)
        detections = net.forward()
        return detections


验证一下人脸检测的效果，其中人脸框上方的`xx%`为置信度

In [ ]:
# 读取测试图片
try:
    img = cv2.imread("test.jpg")
    if img is not None:
        detect = FaceDet()
        detections = detect.detect(img)
        drawed_img = detect.draw_detections(img, detections)
        
        # OpenCV reads image to BGR format. Transform images before showing it.
        drawed_img = cv2.cvtColor(drawed_img, cv2.COLOR_BGR2RGB)
        plt.figure(figsize=(8, 8))
        plt.imshow(drawed_img)
        plt.show()
    else:
        print("test.jpg 文件不存在")
except Exception as e:
    print(f"人脸检测出错: {e}")

## 3. 口罩识别

导入标准库、第三方库，已及MindSpore的模块。

In [ ]:
import math
import os
import numpy as np
import cv2
import matplotlib.pyplot as plt
from easydict import EasyDict
from PIL import Image

from mindspore import context
from mindspore import nn
from mindspore import Tensor
from mindspore.train.model import Model
from mindspore.train.serialization import load_checkpoint, load_param_into_net
from mindspore.train.callback import Callback
from mindspore.train.callback import LossMonitor
from mindspore.train.callback import ModelCheckpoint
from mindspore.train.callback import CheckpointConfig

# 模型定义脚本以及数据处理脚本
from mindspore_py.mobilenetV2 import MobileNetV2Backbone
from mindspore_py.mobilenetV2 import MobileNetV2Head
from mindspore_py.mobilenetV2 import mobilenet_v2
from mindspore_py.dataset import create_dataset

# Log Level = Error
os.environ['GLOG_v'] = '3'
# 设置采用图模式执行，设备为CPU/GPU
context.set_context(mode=context.GRAPH_MODE, device_target="CPU")

### 3.1 数据集介绍

数据信息存放在 `/datasets/5f680a696ec9b83bb0037081-momodel/data/image` 文件夹下。              
收集的图片分成 mask 和 nomask 两类，戴口罩的和没有戴口罩的。        
现在我们尝试读取数据集中的一张戴口罩的图片并显示图片名称。  
现在我们尝试读取数据集中戴口罩的图片及其名称，以下是训练集中的正样本：

In [ ]:
mask_num = 4
fig = plt.figure(figsize=(15, 15))
basic_path = "/home/jovyan/work/datasets/5f680a696ec9b83bb0037081-momodel/data/"
for i in range(mask_num):
    try:
        sub_img = cv2.imread(basic_path + "/image/mask/mask_" + str(i + 101) + ".jpg")
        if sub_img is not None:
            sub_img = cv2.cvtColor(sub_img, cv2.COLOR_BGR2RGB)
            ax = fig.add_subplot(4, 4, (i + 1))
            ax.set_xticks([])
            ax.set_yticks([])
            ax.set_title("mask_" + str(i + 1))
            ax.imshow(sub_img)
        else:
            print(f"图片 mask_{i + 101}.jpg 不存在")
    except Exception as e:
        print(f"读取图片 mask_{i + 101}.jpg 出错: {e}")
plt.show()


以下是训练集中的负样本：

In [ ]:
nomask_num = 4
fig1 = plt.figure(figsize=(15, 15))
for i in range(nomask_num):
    try:
        sub_img = cv2.imread(basic_path + "/image/nomask/nomask_" + str(i + 130) + ".jpg")
        if sub_img is not None:
            sub_img = cv2.cvtColor(sub_img, cv2.COLOR_BGR2RGB)
            ax = fig1.add_subplot(4, 4, (i + 1))
            ax.set_xticks([])
            ax.set_yticks([])
            ax.set_title("nomask_" + str(i + 1))
            ax.imshow(sub_img)
        else:
            print(f"图片 nomask_{i + 130}.jpg 不存在")
    except Exception as e:
        print(f"读取图片 nomask_{i + 130}.jpg 出错: {e}")
plt.show()

### 3.2 模型训练Tips

配置后续训练、验证、推理用到的参数。

In [ ]:
basic_path = "/home/jovyan/work/datasets/5f680a696ec9b83bb0037081-momodel/data/"
config = EasyDict({
    "num_classes": 2,
    "image_height": 224,
    "image_width": 224,
    "data_split": [0.9, 0.1],
    "backbone_out_channels":1280,
    "batch_size": 16,
    "eval_batch_size": 8,
    "epochs": 3,
    "lr_max": 0.01,
    "momentum": 0.9,
    "weight_decay": 1e-4,
    "save_checkpoint": True,
    "save_checkpoint_epochs": 1,
    "save_checkpoint_path": "/home/jovyan/work/results",
    "dataset_path": basic_path + "image",
    "pretrained_ckpt": basic_path + "mindspore_model_data/mobilenetV2-200_1067.ckpt"
})


#### 3.2.1 动态学习率

一般情况下，模型训练时采用静态学习率，如0.01。随着训练步数的增加，模型逐渐趋于收敛，对权重参数的更新幅度应该逐渐降低，以减小模型训练后期的抖动。所以，模型训练时可以采用动态下降的学习率，常见的学习率下降策略有：

- polynomial decay/square decay
- cosine decay
- exponential decay
- stage decay

这里实现 cosine decay 下降策略。

In [ ]:
def cosine_decay(total_steps, lr_init=0.0, lr_end=0.0, lr_max=0.1, warmup_steps=0):
    """
    Applies cosine decay to generate learning rate array.

    Args:
       total_steps(int): all steps in training.
       lr_init(float): init learning rate.
       lr_end(float): end learning rate
       lr_max(float): max learning rate.
       warmup_steps(int): all steps in warmup epochs.

    Returns:
       list, learning rate array.
    """
    lr_init, lr_end, lr_max = float(lr_init), float(lr_end), float(lr_max)
    decay_steps = total_steps - warmup_steps
    lr_all_steps = []
    inc_per_step = (lr_max - lr_init) / warmup_steps if warmup_steps else 0
    for i in range(total_steps):
        if i < warmup_steps:
            lr = lr_init + inc_per_step * (i + 1)  # 修复变量名
        else:
            cosine_decay = 0.5 * (1 + math.cos(math.pi * (i - warmup_steps) / decay_steps))
            lr = (lr_max - lr_end) * cosine_decay + lr_end
        lr_all_steps.append(lr)

    return lr_all_steps

#### 3.2.2 边训练边验证

在面对复杂网络时，往往需要进行几十甚至几百次的epoch训练。在训练之前，很难掌握在训练到第几个epoch时，模型的精度能达到满足要求的程度，所以经常会采用一边训练的同时，在相隔固定epoch的位置对模型进行精度验证，并保存相应的模型，等训练完毕后，通过查看对应模型精度的变化就能迅速地挑选出相对最优的模型。流程如下：

- 定义回调函数EvalCallback，实现同步进行训练和验证。
- 定义训练网络并执行。
- 将不同epoch下的模型精度绘制出折线图并挑选最优模型Checkpoint。

当我们训练深度学习神经网络的时候通常希望能获得最好的泛化性能。但是深度学习神经网络很容易过拟合。当网络在训练集上表现越来越好，错误率越来越低的时候，就极有可能出现了过拟合。我们可以设计一种早停法，比如验证精度连续5次不在上升就停止训练，这样能避免继续训练导致过拟合的问题。

In [ ]:
class EvalCallback(Callback):
    def __init__(self, model, eval_dataset, history, eval_epochs=1):
        self.model = model
        self.eval_dataset = eval_dataset
        self.eval_epochs = eval_epochs
        self.history = history
        self.acc_max = 0
        # acc连续5次<=过程中的最大值，则停止训练
        self.count_max = 5
        self.count = 0

    def epoch_begin(self, run_context):
        self.losses = []

    def step_end(self, run_context):
        cb_param = run_context.original_args()
        loss = cb_param.net_outputs
        if isinstance(loss, (tuple, list)):
            loss = loss[0]
        self.losses.append(loss.asnumpy())

    def epoch_end(self, run_context):
        cb_param = run_context.original_args()
        cur_epoch = cb_param.cur_epoch_num
        train_loss = np.mean(self.losses)

        if cur_epoch % self.eval_epochs == 0:
            metric = self.model.eval(self.eval_dataset, dataset_sink_mode=False)
            self.history["epoch"].append(cur_epoch)
            self.history["eval_acc"].append(metric["acc"])
            self.history["eval_loss"].append(metric["loss"])
            self.history["train_loss"].append(train_loss)
            if self.acc_max < metric["acc"]:
                self.count = 0
                self.acc_max = metric["acc"]
            else:
                self.count += 1
                if self.count == self.count_max:
                    run_context.request_stop()
            print("epoch: %d, train_loss: %f, eval_loss: %f, eval_acc: %f" % (cur_epoch, train_loss, metric["loss"], metric["acc"]))

### 3.3 模型训练

在模型训练过程中，可以添加检查点（Checkpoint）用于保存模型的参数，以便进行推理及中断后再训练使用。使用场景如下：

- 训练后推理场景
    - 模型训练完毕后保存模型的参数，用于推理或预测操作。
    - 训练过程中，通过实时验证精度，把精度最高的模型参数保存下来，用于预测操作。
- 再训练场景
    - 进行长时间训练任务时，保存训练过程中的Checkpoint文件，防止任务异常退出后从初始状态开始训练。
    - Fine-tuning（微调）场景，即训练一个模型并保存参数，基于该模型，面向第二个类似任务进行模型训练。

这里加载 ImageNet 数据上预训练的 MobileNetv2 进行 Fine-tuning，**只训练最后修改的 FC 层**，并在训练过程中保存 Checkpoint。

In [ ]:
import time
import sys
from mindspore import Callback
from mindspore.train.callback import TimeMonitor, LossMonitor

# 动态进度条回调类
class DynamicProgressMonitor(Callback):
    def __init__(self, total_epochs, steps_per_epoch):
        self.total_epochs = total_epochs
        self.steps_per_epoch = steps_per_epoch
        self.current_epoch = 0
        self.current_step = 0
        self.epoch_start_time = None
        
    def on_train_epoch_begin(self, run_context):
        self.epoch_start_time = time.time()
        cb_params = run_context.original_args()
        self.current_epoch = cb_params.cur_epoch_num
        self.current_step = 0
        
        print(f"\nEpoch {self.current_epoch}/{self.total_epochs}")
        print("-" * 60)
        
    def on_train_step_begin(self, run_context):
        self.current_step += 1
        
    def on_train_step_end(self, run_context):
        # 计算进度
        progress = self.current_step / self.steps_per_epoch
        bar_length = 40
        filled_length = int(bar_length * progress)
        
        # 创建进度条
        bar = '█' * filled_length + '░' * (bar_length - filled_length)
        
        # 计算预计剩余时间
        elapsed_time = time.time() - self.epoch_start_time
        if progress > 0:
            estimated_total = elapsed_time / progress
            remaining_time = estimated_total - elapsed_time
            time_str = f"{remaining_time:.0f}s"
        else:
            time_str = "?"
            
        # 动态更新进度条
        sys.stdout.write(f'\r  [{bar}] {self.current_step}/{self.steps_per_epoch} ({progress:.1%}) | ETA: {time_str}')
        sys.stdout.flush()
        
    def on_train_epoch_end(self, run_context):
        epoch_time = time.time() - self.epoch_start_time
        print(f'\n  ✓ 完成! 耗时: {epoch_time:.2f}s')
        
    def on_train_end(self, run_context):
        print("\n" + "="*60)
        print("训练完成!")

# 运行该 cell 代码训练模型时，请清理 results 文件夹中 mindspore 框架之前训练好的模型，否则 model_path 格式会发生一定的变化，造成推理时可能找不到模型的报错
def train():
    train_dataset, eval_dataset = create_dataset(dataset_path=config.dataset_path, config=config)
    step_size = train_dataset.get_dataset_size()

    backbone = MobileNetV2Backbone()
    # Freeze parameters of backbone. You can comment these two lines.
    for param in backbone.get_parameters():
       param.requires_grad = False
    # load parameters from pretrained model
    load_checkpoint(config.pretrained_ckpt, backbone)

    head = MobileNetV2Head(input_channel=backbone.out_channels, num_classes=config.num_classes)
    network = mobilenet_v2(backbone, head)

    # define loss, optimizer, and model
    loss = nn.SoftmaxCrossEntropyWithLogits(sparse=True, reduction='mean')
    lrs = cosine_decay(config.epochs * step_size, lr_max=config.lr_max)
    opt = nn.Momentum(network.trainable_params(), lrs, config.momentum, config.weight_decay)
    model = Model(network, loss, opt, metrics={'acc', 'loss'})

    history = {'epoch': [], 'train_loss': [], 'eval_loss': [], 'eval_acc': []}
    eval_cb = EvalCallback(model, eval_dataset, history)
    
    # 添加动态进度条回调
    progress_cb = DynamicProgressMonitor(config.epochs, step_size)
    time_cb = TimeMonitor()
    loss_cb = LossMonitor()
    
    cb = [eval_cb, progress_cb, time_cb, loss_cb]
    
    if config.save_checkpoint:
        ckpt_cfg = CheckpointConfig(save_checkpoint_steps=config.save_checkpoint_epochs * step_size, keep_checkpoint_max=config.epochs)
        ckpt_cb = ModelCheckpoint(prefix="mobilenetv2_mask", directory=config.save_checkpoint_path, config=ckpt_cfg)
        cb.append(ckpt_cb)
    
    print("="*60)
    print("开始训练口罩佩戴检测模型")
    print("="*60)
    print(f"总epoch数: {config.epochs}")
    print(f"每个epoch的step数: {step_size}")
    print(f"总step数: {config.epochs * step_size}")
    print("="*60)
    
    model.train(config.epochs, train_dataset, callbacks=cb, dataset_sink_mode=False)

    return history

将不同 epoch 下的模型精度绘制出折线图并挑选最优模型 Checkpoint。

In [ ]:
import os
import mindspore.dataset as ds
from mindspore.dataset.vision import c_transforms as C
from mindspore.dataset.transforms import c_transforms as C2
import mindspore.common.dtype as mstype

def create_filtered_dataset(data_path, batch_size=32, repeat_num=1, training=True, num_parallel_workers=1):
    """
    创建过滤后的数据集，排除隐藏文件和目录
    """
    # 定义数据增强操作
    if training:
        transform = [
            C.RandomCropDecodeResize(224, scale=(0.08, 1.0), ratio=(0.75, 1.333)),
            C.RandomHorizontalFlip(prob=0.5),
            C.RandomColorAdjust(brightness=0.4, contrast=0.4, saturation=0.4),
            C.Normalize(mean=[127.5, 127.5, 127.5], std=[127.5, 127.5, 127.5]),
            C.HWC2CHW()
        ]
    else:
        transform = [
            C.Decode(),
            C.Resize(256),
            C.CenterCrop(224),
            C.Normalize(mean=[127.5, 127.5, 127.5], std=[127.5, 127.5, 127.5]),
            C.HWC2CHW()
        ]
    
    type_cast_op = C2.TypeCast(mstype.int32)
    
    # 使用自定义文件列表而不是ImageFolderDataset
    def get_filtered_file_list(data_path):
        image_extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.JPG', '.JPEG', '.PNG', '.BMP'}
        file_list = []
        label_list = []
        
        for label, class_name in enumerate(['mask', 'nomask']):
            class_path = os.path.join(data_path, class_name)
            if not os.path.exists(class_path):
                continue
                
            for filename in os.listdir(class_path):
                # 跳过隐藏文件和目录
                if filename.startswith('.') or filename.startswith('_'):
                    continue
                    
                file_path = os.path.join(class_path, filename)
                # 只处理文件，不处理目录
                if os.path.isfile(file_path):
                    # 检查文件扩展名
                    _, ext = os.path.splitext(filename)
                    if ext.lower() in image_extensions:
                        file_list.append(file_path)
                        label_list.append(label)
        
        return file_list, label_list
    
    # 获取过滤后的文件列表
    file_list, label_list = get_filtered_file_list(data_path)
    print(f"找到 {len(file_list)} 个有效图像文件")
    
    # 创建数据集
    dataset = ds.NumpySlicesDataset({'image': file_list, 'label': label_list}, 
                                   column_names=['image', 'label'], 
                                   shuffle=training)
    
    # 应用解码和变换
    dataset = dataset.map(operations=C.Decode(), input_columns="image", 
                         num_parallel_workers=num_parallel_workers)
    dataset = dataset.map(operations=transform, input_columns="image", 
                         num_parallel_workers=num_parallel_workers)
    dataset = dataset.map(operations=type_cast_op, input_columns="label", 
                         num_parallel_workers=num_parallel_workers)
    
    # 设置batch_size
    dataset = dataset.batch(batch_size, drop_remainder=True)
    
    return dataset

In [ ]:
history = train()

plt.plot(history['epoch'], history['train_loss'], label='train_loss')
plt.plot(history['epoch'], history['eval_loss'], 'r', label='val_loss')
plt.legend()
plt.show()

plt.plot(history['epoch'], history['eval_acc'], 'r', label = 'val_acc')
plt.legend()
plt.show()

model_path = '/home/jovyan/work/results/mobilenetv2_mask-%d_39.ckpt' % (np.argmax(history['eval_acc']) + 1) # 挑选出最优模型Checkpoint
print("the path of best model checkpoint is :", model_path)


### 3.4 模型推理

加载模型 Checkpoint 进行推理。           
使用 load_checkpoint 接口加载数据时，需要把数据传入给原始网络，而不能传递给带有优化器和损失函数的训练网络。

In [ ]:
def image_process(image):
    """Precess one image per time.

    Args:
        image: shape (H, W, C)
    """
    mean = [0.485 * 255, 0.456 * 255, 0.406 * 255]
    std = [0.229 * 255, 0.224 * 255, 0.225 * 255]
    image = (np.array(image) - mean) / std
    image = image.transpose((2, 0, 1))
    img_tensor = Tensor(np.array([image], np.float32))
    return img_tensor

def infer_one(network, image_path):
    try:
        image = Image.open(image_path).resize((config.image_height, config.image_width))
        logits = network(image_process(image))
        pred = np.argmax(logits.asnumpy(), axis=1)[0]
        print("图片路径：", image_path, "图片预测类别", pred)
    except Exception as e:
        print(f"推理图片 {image_path} 时出错: {e}")

def infer(basic_path, model_path):
    if model_path is None or not os.path.exists(model_path):
        print("模型路径无效或文件不存在")
        return
        
    backbone = MobileNetV2Backbone(last_channel=config.backbone_out_channels)
    head = MobileNetV2Head(input_channel=backbone.out_channels, num_classes=config.num_classes)
    network = mobilenet_v2(backbone, head)
    load_checkpoint(model_path, network)
    
    # 测试戴口罩的图片
    print("=== 戴口罩图片测试 ===")
    for i in range(250, 258):
        infer_one(network, basic_path + 'image/mask/mask_%s.jpg' % i)
    
    # 测试不戴口罩的图片
    print("=== 不戴口罩图片测试 ===")
    for i in range(371, 378):
        infer_one(network, basic_path + 'image/nomask/nomask_%s.jpg' % i)

In [ ]:
if model_path and os.path.exists(model_path):
    infer(basic_path, model_path)
else:
    print("无法执行推理：模型路径无效")

### 3.5 口罩识别

In [ ]:
class MaskRec():
    def __init__(self, model_path):
        self.face_det = FaceDet()
        self.mask_model_input_size = (160, 160)
        self.class_names = ['YES', 'NO']
        
        # 初始化MobileNetv2
        backbone = MobileNetV2Backbone(last_channel=config.backbone_out_channels)
        head = MobileNetV2Head(input_channel=backbone.out_channels, num_classes=config.num_classes)
        self.mask_model = mobilenet_v2(backbone, head)
        
        if model_path and os.path.exists(model_path):
            load_checkpoint(model_path, self.mask_model)
            print("口罩识别模型加载成功")
        else:
            print("警告：模型路径无效，使用未训练的模型")

    def to_small_square(self, startX, startY, endX, endY):
        w = endX - startX
        h = endY - startY
        l = min(w, h)

        startX = int(startX + (w - l) / 2)
        endX = startX + l
        startY = int(startY + (h - l) / 2)
        endY = startY + l
        return startX, startY, endX, endY

    def recognize(self, image):
        # 人脸检测
        detections = self.face_det.detect(image)
        h, w, c = image.shape
        predict_labels = []
        
        for i in range(0, detections.shape[2]):
            confidence = detections[0, 0, i, 2]
            if confidence > self.face_det.threshold:
                box = detections[0, 0, i, 3:7] * np.array([w, h, w, h])
                startX, startY, endX, endY = box.astype("int")
                
                # 确保坐标在图像范围内
                startX, startY = max(0, startX), max(0, startY)
                endX, endY = min(w, endX), min(h, endY)
                
                if endX <= startX or endY <= startY:
                    continue
                    
                # 截取图像
                startX, startY, endX, endY = self.to_small_square(startX, startY, endX, endY)
                
                # 再次检查坐标
                startX, startY = max(0, startX), max(0, startY)
                endX, endY = min(w, endX), min(h, endY)
                
                if endX <= startX or endY <= startY:
                    continue
                    
                try:
                    crop_img = image[startY:endY, startX:endX]
                    if crop_img.size == 0:
                        continue
                        
                    # 图像预处理, mask_model accept image with RGB
                    resized = cv2.resize(crop_img, (config.image_height, config.image_width))
                    img_tensor = image_process(cv2.cvtColor(resized, cv2.COLOR_BGR2RGB))
                    
                    # 预测
                    logits = self.mask_model(img_tensor)
                    predict_label = np.argmax(logits.asnumpy(), axis=1)[0]
                    predict_labels.append(predict_label)
                    
                    # 画图
                    y = startY - 10 if startY - 10 > 10 else startY + 10
                    cv2.rectangle(image, (startX, startY), (endX, endY), (0, 255, 0), 2)
                    cv2.putText(image, self.class_names[predict_label], (startX, y),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 255, 0), 2)
                except Exception as e:
                    print(f"处理人脸时出错: {e}")
                    continue
                    
        return image, len(predict_labels), predict_labels.count(0)

In [ ]:
try:
    img = cv2.imread("./test1.jpg")
    if img is not None:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        if model_path and os.path.exists(model_path):
            detect = MaskRec(model_path)
            img, all_num, mask_num = detect.recognize(img)
            
            # 展示图片口罩识别结果
            fig = plt.figure(figsize=(8, 8))
            ax1 = fig.add_subplot(111)
            ax1.set_xticks([])
            ax1.set_yticks([])
            ax1.set_title('口罩识别结果')
            ax1.imshow(img)
            plt.show()
            
            print("图中的人数有：" + str(all_num) + "个")
            print("戴口罩的人数有：" + str(mask_num) + "个")
        else:
            print("无法进行口罩识别：模型文件不存在")
    else:
        print("test1.jpg 文件不存在")
except Exception as e:
    print(f"口罩识别测试出错: {e}")

## 4. 实训总结
本实训基于 MindSpore 框架实现了一个高效的单阶段目标检测方案，完成了 OpenCV DNN 模块的人脸检测和 MobileNetV2 分类网络的集成。我们深入学习了基于 SSD 架构的目标检测原理，掌握了先验框设计和多尺度特征融合的核心思想。

通过实践，我们理解了国产深度学习框架在目标检测任务中的优势，学会了动态学习率调整、早停机制等模型优化策略。特别地，我们掌握了将传统计算机视觉库与深度学习框架相结合的技术路径，为在实际工程中部署轻量级目标检测系统积累了宝贵经验。